In [7]:
!pip install -q -U groq langgraph langchain-core

In [6]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("aaditi21")

if not GROQ_API_KEY:
    raise ValueError("Groq API key not found.")

print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [8]:
from groq import Groq

client = Groq(
    api_key=GROQ_API_KEY
)

MODEL_NAME = "openai/gpt-oss-120b"

print("Groq client initialized successfully!")
print("Model:", MODEL_NAME)

Groq client initialized successfully!
Model: openai/gpt-oss-120b


In [9]:
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Explain LangGraph in simple words."
        }
    ],
    temperature=0.2
)

print(response.choices[0].message.content)

**LangGraph in a nutshell**

Imagine you have a robot that can talk, think, and do little tasks (like looking up information, writing a summary, or asking you a follow‑up question). Instead of giving the robot a single, long set of instructions, you break the work into **small steps** and tell the robot how those steps connect to each other.  

That’s exactly what **LangGraph** does for language‑model (LLM) applications:

| What it is | Simple analogy |
|------------|----------------|
| **A Python library** for building “graphs” of LLM‑driven actions. | Like drawing a flowchart that shows what the robot does first, then next, and so on. |
| **Nodes** = individual pieces of work (e.g., “call the model”, “search the web”, “store a result”). | Think of each node as a tiny worker with a specific job. |
| **Edges** = the rules that decide which node runs after another (maybe based on the model’s answer). | Like the arrows on a flowchart that say “if the answer is yes, go here; otherwise, go

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. Define the Graph State
class AgentState(TypedDict):
    user_query: str
    response: str
    is_valid: bool

In [11]:
def generate_response_node(state: AgentState):

    query = state["user_query"]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0.2
    )

    answer = response.choices[0].message.content

    return {
        "response": answer
    }

In [12]:
def validation_node(state: AgentState):

    valid = len(state["response"]) > 10

    return {
        "is_valid": valid
    }

In [13]:
def router(state: AgentState):

    if state["is_valid"]:
        return "approved"
    else:
        return "rejected"

In [14]:
builder = StateGraph(AgentState)

# Add nodes
builder.add_node("generator", generate_response_node)
builder.add_node("validator", validation_node)

# Starting point
builder.add_edge(START, "generator")

# Generator → Validator
builder.add_edge("generator", "validator")

# Conditional routing
builder.add_conditional_edges(
    "validator",
    router,
    {
        "approved": END,
        "rejected": "generator"
    }
)

# Compile
app = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [15]:
output = app.invoke({
    "user_query": "Explain what LangGraph is in simple words."
})

print("--- LANGGRAPH EXECUTION COMPLETE ---")
print("Final Response:")
print(output["response"])

--- LANGGRAPH EXECUTION COMPLETE ---
Final Response:
**LangGraph in a nutshell**

Imagine you want to build a chatbot or any AI‑powered tool that can do many steps—like asking a question, looking up information, deciding what to do next, and then giving a final answer. LangGraph is a Python library that helps you **connect those steps together like a flowchart or a graph**, so the AI can move back and forth between them, remember what happened, and make decisions along the way.

### Key ideas, broken down simply

| Concept | What it means in plain language |
|---------|-----------------------------------|
| **Graph of “nodes”** | Think of each node as a small task (e.g., “ask the user for details”, “search the web”, “summarize results”). The graph shows how you can jump from one task to another. |
| **State** | A notebook that the graph carries around, storing things like the user’s input, intermediate results, or flags that tell the next step what to do. |
| **Edges (transitions)** | 

In [17]:
#MCP
# Install required packages
!pip install -q -U groq langchain-groq langchain-core langgraph

import os
import csv
import asyncio
import nest_asyncio

from google.colab import userdata

# LangChain & LangGraph Imports
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

nest_asyncio.apply()


# 1. Setup Groq API Key from Colab Secrets
GROQ_API_KEY = userdata.get("aaditi21")

if not GROQ_API_KEY:
    raise ValueError("Groq API key not found.")


# 2. Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=GROQ_API_KEY,
    temperature=0
)


# -------------------------------------------------------------
# 3. Local Real-World Tools (MCP Primitive Logic)
# -------------------------------------------------------------

CSV_FILE = "expenses.csv"


def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])


def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""

    _initialize_csv()

    with open(CSV_FILE, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])

    return f"Successfully logged expense: {item} - ₹{amount} ({category})"


def get_expenses() -> str:
    """Retrieves all logged expenses from the CSV file."""

    _initialize_csv()

    with open(CSV_FILE, mode="r") as f:
        reader = csv.reader(f)
        rows = list(reader)

    if len(rows) <= 1:
        return "No expenses recorded yet."

    return "\n".join([", ".join(row) for row in rows])


# -------------------------------------------------------------
# 4. Wrap Tools as LangChain Structured Tools
# -------------------------------------------------------------

mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),

    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]


# -------------------------------------------------------------
# 5. Build & Execute Agent
# -------------------------------------------------------------

agent = create_react_agent(llm, mcp_tools)


async def run_agentic_workflow():

    print("--- Task 1: Log an expense ---")

    prompt_1 = "I bought a pizza for ₹250. Category is Food."

    response_1 = await agent.ainvoke({
        "messages": [
            ("user", prompt_1)
        ]
    })

    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")


    print("\n--- Task 2: Retrieve records ---")

    prompt_2 = "Show me all expenses logged so far."

    response_2 = await agent.ainvoke({
        "messages": [
            ("user", prompt_2)
        ]
    })

    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")


# Run Execution Loop
asyncio.run(run_agentic_workflow())



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.6 MB/s eta 0:00:00


/tmp/ipykernel_2761/1109450916.py:98: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)


--- Task 1: Log an expense ---

[Agent Response]: Got it! I’ve logged your pizza purchase:

- **Item:** Pizza  
- **Amount:** ₹250  
- **Category:** Food  

Let me know if there’s anything else you’d like to add or review.

--- Task 2: Retrieve records ---

[Agent Response]: Here are the expenses that have been logged so far:

| Item  | Amount (₹) | Category |
|-------|------------|----------|
| pizza | 250.0      | Food |
